In [3]:
import csv
import json
import os

import datasets


_CITATION = """\
@InProceedings{ealvaradob:dataset,
title = {Phishing Datasets},
author={Esteban Alvarado},
year={2024}
}
"""

_DESCRIPTION = """\
Dataset designed for phishing classification tasks in various data types.
"""
_HOMEPAGE = ""

_LICENSE = ""

_URLS = {
    "texts": "texts.json",
    "urls": "urls.json",
    "webs": "webs.json",
    "combined_full": "combined_full.json",
    "combined_reduced": "combined_reduced.json"
}

In [7]:
class PhishingDatasets(datasets.GeneratorBasedBuilder):
    """Phishing Datasets Configuration"""

    VERSION = datasets.Version("1.1.0")

    BUILDER_CONFIGS = [
        datasets.BuilderConfig(name="texts", version=VERSION, description="text subset"),
        datasets.BuilderConfig(name="urls", version=VERSION, description="urls subset"),
        datasets.BuilderConfig(name="webs", version=VERSION, description="webs subset"),
        datasets.BuilderConfig(name="combined_full", version=VERSION, description="combined dataset that have all URLs"),
        datasets.BuilderConfig(name="combined_reduced", version=VERSION, description="combined dataset that doesn't have all URLs for representativity issues"),
    ]

    DEFAULT_CONFIG_NAME = "combined_reduced"

    def _info(self):
        features = datasets.Features(
            {
                "text": datasets.Value("string"),
                "label": datasets.Value("int64"),
            }
        )
        return datasets.DatasetInfo(
            description=_DESCRIPTION,
            features=features,
            supervised_keys=("text", "label"),
            homepage=_HOMEPAGE,
            license=_LICENSE,
            citation=_CITATION,
        )

    def _split_generators(self, dl_manager):
        urls = _URLS[self.config.name]
        data_dir = dl_manager.download_and_extract(urls)
        return [
            datasets.SplitGenerator(
                name=datasets.Split.TRAIN,
                gen_kwargs={
                    "filepath": data_dir,
                    "split": "train",
                },
            ),
        ]

    def _generate_examples(self, filepath, split):
        with open(filepath, encoding="utf-8") as f:
            data = json.load(f)
            for index, sample in enumerate(data):
                yield index, {
                    "text": sample['text'],
                    "label": sample['label']
                }

# Przykład ładowania i wyświetlenia danych (zakładając, że pliki JSON znajdują się w Twoim katalogu roboczym):
# dataset = datasets.load_dataset("./nazwa_pliku_skryptu.py", name="combined_reduced")
# print(dataset["train"][0])

In [9]:
from datasets import Dataset
from sklearn.model_selection import train_test_split
from pathlib import Path
import os
import urllib.request
import json

# Ponieważ zdalne skrypty datasetów są wyłączone, pobieramy bezpośrednio plik JSON z repozytorium Hugging Face
# (z lokalnym cache w data/combined_reduced.json, aby kolejne uruchomienia działały w 100% offline):
url = "https://huggingface.co/datasets/ealvaradob/phishing-dataset/resolve/main/combined_reduced.json"
cache_path = Path("data/combined_reduced.json")
cache_path.parent.mkdir(parents=True, exist_ok=True)

if not cache_path.exists():
    response = urllib.request.urlopen(url)
    raw_bytes = response.read()
    cache_path.write_bytes(raw_bytes)
    data = json.loads(raw_bytes.decode("utf-8"))
else:
    with open(cache_path, "r", encoding="utf-8") as f:
        data = json.load(f)

# Tworzymy obiekt Dataset bezpośrednio z pobranych danych
df_data = {"text": [item["text"] for item in data], "label": [item["label"] for item in data]}
dataset = Dataset.from_dict(df_data)

df = dataset.to_pandas()
train, test = train_test_split(df, test_size=0.2, random_state=42)
train, test = Dataset.from_pandas(train, preserve_index=False), Dataset.from_pandas(test, preserve_index=False)

In [10]:
len(train)

62141

---
### Architektura zoptymalizowana pod 3 metryki:
1. **Wielkość rozwiązania (`< 400 KB`)**: bezstanowy `HashingVectorizer` (słowny `1–2` + znakowy `char_wb` `3–4`, po $2^{15}$ kubłów, zero słownika na dysku) + wektor wag `float32` z kompresją `zlib`.
2. **Szybkość predykcji (`< 1 s` dla 15 536 próbek)**: przycinanie surowego HTML do 4096 znaków (`O(1)`) + pojedyncze mnożenie rzadkiej macierzy przez wektor w `SGDClassifier(loss='log_loss')`.
3. **Czułość predykcji (`Recall`)**: 14 cech strukturalno-leksykalnych cybersecurity (entropia Shannona, IPv4, punycode `xn--`, `@`, `<input type='password'>`, obfuskacja JS `eval/atob`, słowa presji) + kalibracja progu decyzyjnego $\tau$ na wewnętrznej walidacji.

In [ ]:
import time
from pathlib import Path
import joblib
import numpy as np
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

# Gwarantuje dostępność modułu phishing_features.py nawet przy uruchomieniu samego notatnika (np. w Google Colab)
if not Path("phishing_features.py").exists():
    Path("phishing_features.py").write_text('"""\nModuł ekstrakcji cech i budowy potoku dla lokalnego filtra antyphishingowego (Opcja A).\nWspółdzielony pomiędzy notatnikiem `phishing_detector.ipynb` a skryptem `predict.py`,\nco gwarantuje bezbłędną serializację i deserializację modelu (.joblib) bez zależności od __main__.\n"""\n\nfrom __future__ import annotations\n\nimport math\nimport re\nfrom collections import Counter\nfrom typing import Iterable, List\n\nimport numpy as np\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.feature_extraction.text import HashingVectorizer\nfrom sklearn.linear_model import SGDClassifier\nfrom sklearn.pipeline import FeatureUnion, Pipeline\nfrom sklearn.preprocessing import FunctionTransformer, MaxAbsScaler\n\n# Przycięcie tekstu wejściowego do 4096 znaków gwarantuje czas O(1) dla dużych stron HTML (>100 KB)\nMAX_TEXT_LEN_FAST = 4_096\n\n_IPV4_RE = re.compile(r"(?:https?://)?(?:\\d{1,3}\\.){3}\\d{1,3}(?::\\d+)?(?:/|$)")\n_PWD_INPUT_RE = re.compile(r"<input[^>]+type\\s*=\\s*[\'\\"]?password", re.IGNORECASE)\n_JS_OBF_RE = re.compile(\n    r"(?:eval\\s*\\(|atob\\s*\\(|fromcharcode|document\\.write\\s*\\()",\n    re.IGNORECASE,\n)\n_URGENCY_RE = re.compile(\n    r"\\b(?:verify|suspend(?:ed)?|urgent|account|password|login|sign[\\s-]?in|"\n    r"confirm|security|update|bank|wallet|expired|billing|invoice)\\b",\n    re.IGNORECASE,\n)\n_HTML_TAG_RE = re.compile(r"(?:<!doctype|<html|<head|<body|<form|<script)", re.IGNORECASE)\n\n\ndef fast_truncate(texts: Iterable[str], max_len: int = MAX_TEXT_LEN_FAST) -> List[str]:\n    """Przycina ciąg wejściowy do ustalonego prefiksu w czasie O(1)."""\n    return [t[:max_len] if isinstance(t, str) else "" for t in texts]\n\n\ndef _shannon_entropy_fast(sample: str) -> float:\n    """Oblicza entropię Shannona na krótkim prefiksie (wykrywanie domen DGA i ciągów Base64)."""\n    if not sample:\n        return 0.0\n    counts = Counter(sample)\n    inv_len = 1.0 / len(sample)\n    return -sum((c * inv_len) * math.log2(c * inv_len) for c in counts.values())\n\n\nclass FastLexicalExtractor(BaseEstimator, TransformerMixin):\n    """\n    Ultraszybki ekstraktor 14 cech leksykalnych i strukturalnych (float32).\n    Operuje wyłącznie na krótkich prefiksach ciągu wejściowego (URL / HTML / SMS).\n    """\n\n    FEATURE_NAMES = [\n        "log_len",\n        "is_url_only",\n        "is_html",\n        "digit_ratio",\n        "special_char_ratio",\n        "shannon_entropy",\n        "has_ipv4",\n        "has_punycode",\n        "at_symbol_count",\n        "url_depth_slashes",\n        "subdomain_dots",\n        "has_password_input",\n        "has_js_obfuscation",\n        "urgency_keyword_hits",\n    ]\n\n    def __init__(self, max_len: int = MAX_TEXT_LEN_FAST):\n        self.max_len = max_len\n\n    def fit(self, X: Iterable[str], y=None):\n        return self\n\n    def transform(self, X: Iterable[str]) -> np.ndarray:\n        rows = []\n        for raw in X:\n            s = raw[: self.max_len] if isinstance(raw, str) else ""\n            n = len(s)\n            if n == 0:\n                rows.append([0.0] * 14)\n                continue\n\n            head_120 = s[:120]\n            head_256 = s[:256]\n            lower_head_256 = head_256.lower()\n            lower_s = s.lower()\n\n            log_len = math.log1p(n)\n            stripped = s.strip()\n            is_url_only = float(\n                stripped.startswith(("http://", "https://", "www."))\n                and ("\\n" not in stripped)\n                and (" " not in stripped)\n            )\n            is_html = float(bool(_HTML_TAG_RE.search(head_256)))\n\n            digits = sum(ch.isdigit() for ch in head_256)\n            specials = sum((not ch.isalnum()) and (not ch.isspace()) for ch in head_256)\n            inv_256 = 1.0 / len(head_256)\n\n            digit_ratio = digits * inv_256\n            special_char_ratio = specials * inv_256\n            entropy = _shannon_entropy_fast(head_256)\n\n            has_ipv4 = float(bool(_IPV4_RE.search(head_256)))\n            has_punycode = float("xn--" in lower_head_256)\n            at_count = float(head_256.count("@"))\n            slash_count = float(head_256.count("/"))\n            dot_count = float(head_120.count("."))\n\n            has_pwd = float(bool(_PWD_INPUT_RE.search(lower_s)))\n            has_js_obf = float(bool(_JS_OBF_RE.search(lower_s)))\n            urgency_hits = float(min(len(_URGENCY_RE.findall(s[:1500])), 12))\n\n            rows.append(\n                [\n                    log_len,\n                    is_url_only,\n                    is_html,\n                    digit_ratio,\n                    special_char_ratio,\n                    entropy,\n                    has_ipv4,\n                    has_punycode,\n                    at_count,\n                    slash_count,\n                    dot_count,\n                    has_pwd,\n                    has_js_obf,\n                    urgency_hits,\n                ]\n            )\n\n        return np.asarray(rows, dtype=np.float32)\n\n\ndef build_option_a_pipeline(\n    n_features_word: int = 2**15,\n    n_features_char: int = 2**15,\n    alpha: float = 1e-5,\n    random_state: int = 42,\n) -> Pipeline:\n    """\n    Buduje bezstanowy, ultralekki potok Opcji A (Ultra-Fast & Ultra-Compact):\n    1. Przycięcie wejścia do 4096 znaków (fast_truncate).\n    2. Równoległa wektoryzacja haszująca (słowna 1-2 + znakowa char_wb 3-4) + 14 cech strukturalnych.\n    3. Liniowy klasyfikator logistyczny SGDClassifier(loss=\'log_loss\').\n    """\n    features = FeatureUnion(\n        transformer_list=[\n            (\n                "word_hash",\n                HashingVectorizer(\n                    n_features=n_features_word,\n                    ngram_range=(1, 2),\n                    token_pattern=r"(?u)\\b\\w\\w+\\b",\n                    alternate_sign=False,\n                    norm="l2",\n                    dtype=np.float32,\n                ),\n            ),\n            (\n                "char_hash",\n                HashingVectorizer(\n                    n_features=n_features_char,\n                    analyzer="char_wb",\n                    ngram_range=(3, 4),\n                    alternate_sign=False,\n                    norm="l2",\n                    dtype=np.float32,\n                ),\n            ),\n            (\n                "struct",\n                Pipeline(\n                    [\n                        ("extract", FastLexicalExtractor(max_len=MAX_TEXT_LEN_FAST)),\n                        ("scale", MaxAbsScaler()),\n                    ]\n                ),\n            ),\n        ]\n    )\n\n    clf = SGDClassifier(\n        loss="log_loss",\n        penalty="l2",\n        alpha=alpha,\n        max_iter=100,\n        tol=1e-3,\n        class_weight="balanced",\n        random_state=random_state,\n    )\n\n    return Pipeline(\n        [\n            ("truncate", FunctionTransformer(fast_truncate, validate=False)),\n            ("features", features),\n            ("clf", clf),\n        ]\n    )\n', encoding="utf-8")

from phishing_features import build_option_a_pipeline

def calibrate_threshold_for_recall(y_true: np.ndarray, y_proba: np.ndarray, target_recall: float = 0.94) -> float:
    """Dobiera najwyższy próg decyzyjny tau gwarantujący czułość >= target_recall."""
    _, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    valid_idx = np.where(recalls[:-1] >= target_recall)[0]
    if len(valid_idx) == 0:
        return 0.30
    return float(np.clip(thresholds[valid_idx[-1]], 0.15, 0.65))

print(f"Zbiór treningowy (Dataset): {len(train):,} próbek | Zbiór testowy (Dataset): {len(test):,} próbek")

In [ ]:
# 1. Kalibracja progu tau na wewnętrznej próbie walidacyjnej (10% zbioru `train`, bez wycieku ze zbioru `test`)
TARGET_RECALL = 0.94
val_split = train.train_test_split(test_size=0.1, seed=42)

t_train_start = time.perf_counter()
cal_pipeline = build_option_a_pipeline()
cal_pipeline.fit(val_split["train"]["text"], np.asarray(val_split["train"]["label"]))
val_proba = cal_pipeline.predict_proba(val_split["test"]["text"])[:, 1]
tau = calibrate_threshold_for_recall(
    np.asarray(val_split["test"]["label"]), val_proba, target_recall=TARGET_RECALL
)

# 2. Trening finalnego modelu Opcji A na pełnym zbiorze `train` (62 141 rekordów)
model_pipeline = build_option_a_pipeline()
model_pipeline.fit(train["text"], np.asarray(train["label"]))

# Kompresja wektora wag do float32 przed zapisem na dysk
model_pipeline.named_steps["clf"].coef_ = model_pipeline.named_steps["clf"].coef_.astype(np.float32)
model_pipeline.named_steps["clf"].intercept_ = model_pipeline.named_steps["clf"].intercept_.astype(np.float32)
train_time_sec = time.perf_counter() - t_train_start

# 3. Zapis lokalnego artefaktu modelu (.joblib)
model_path = Path("models/phishing_detector_option_a.joblib")
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(
    {
        "pipeline": model_pipeline,
        "threshold": tau,
        "option": "A_ultra_fast",
        "target_recall": TARGET_RECALL,
        "dataset_source": "ealvaradob/phishing-dataset:combined_reduced",
    },
    model_path,
    compress=3,
)

print(f"Trening zakończony w {train_time_sec:.2f} s | Skalibrowany próg tau = {tau:.4f}")
print(f"Zapisano lokalny model: {model_path} ({model_path.stat().st_size / 1024:.2f} KB)")

In [ ]:
# 4. Ewaluacja 3 kluczowych metryk na zbiorze `test` (20% = 15 536 rekordów)
test_texts = test["text"]
y_test = np.asarray(test["label"])

# Warm-up 10 próbek
_ = model_pipeline.predict_proba(test_texts[:10])

t_inf_start = time.perf_counter()
test_proba = model_pipeline.predict_proba(test_texts)[:, 1]
y_pred = (test_proba >= tau).astype(int)
inf_time_sec = time.perf_counter() - t_inf_start

model_size_kb = model_path.stat().st_size / 1024.0
scripts_size_kb = sum(
    Path(p).stat().st_size for p in ["phishing_features.py", "predict.py"] if Path(p).exists()
) / 1024.0
recall_val = recall_score(y_test, y_pred)
precision_val = precision_score(y_test, y_pred)
f1_val = f1_score(y_test, y_pred)
auc_val = roc_auc_score(y_test, test_proba)

print("=" * 66)
print("WYNIKI FILTRA ANTYPHISHINGOWEGO (OPCJA A) NA ZBIORZE `test`")
print("=" * 66)
print(f"1. Wielkość rozwiązania : model = {model_size_kb:.2f} KB | skrypty = {scripts_size_kb:.2f} KB")
print(f"2. Szybkość predykcji   : {inf_time_sec:.4f} s dla {len(test):,} próbek "
      f"({(inf_time_sec / len(test)) * 1000:.4f} ms/próbkę | {len(test) / inf_time_sec:,.1f} próbek/s)")
print(f"3. Czułość (Recall)     : {recall_val * 100:.2f}% "
      f"(Precision: {precision_val * 100:.2f}% | F1: {f1_val * 100:.2f}% | ROC-AUC: {auc_val:.4f})")
print(f"   Macierz pomyłek [[TN, FP], [FN, TP]]: {confusion_matrix(y_test, y_pred).tolist()}")
print("=" * 66)
print("\nSzczegółowy raport klasyfikacji:\n")
print(classification_report(y_test, y_pred, target_names=["Benign (0)", "Phishing (1)"], digits=4))

# Zapis 100 próbek testowych do szybkich testów lokalnych skryptem predict.py
sample_out = Path("data/test_sample_100.json")
sample_records = [{"text": test_texts[i], "label": int(y_test[i])} for i in range(min(100, len(test)))]
sample_out.write_text(json.dumps(sample_records, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Zapisano paczkę 100 próbek testowych do: {sample_out}")